# NYSC youth survey: cleaning and transformation

This notebook converts the flattened survey CSV into analysis-ready and clustering-ready datasets for the NYSC youth-transition study.

It follows the preparation plan in the supervisor document:

- preserve the order of ordinal variables;
- one-hot encode nominal variables instead of assigning arbitrary numeric ranks;
- distinguish genuine missing responses from branch-driven **not applicable** values;
- impute only after data-quality filtering;
- standardize numeric features;
- retain PCA components explaining approximately 85% of variance;
- prepare both a full-sample matrix and separate D/E/F/G branch matrices.

The Data Analysis Report is a reporting template rather than a dataset-specific specification. This notebook therefore includes auditable outputs for data cleaning, missing-data handling, encoding, feature engineering, outlier diagnostics, scaling, and dimensionality reduction.

> Scope: this notebook prepares data for clustering. It deliberately does not select a clustering algorithm or assign cluster labels.

In [1]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

INPUT_CSV = Path("nysc_survey.survey_responses23.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
PCA_VARIANCE_TARGET = 0.85
MIN_ANSWERED_FEATURES = 10

assert INPUT_CSV.exists(), f"Input file not found: {INPUT_CSV.resolve()}"
print("Input:", INPUT_CSV.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Input: /Users/vimashafernando/Desktop/ClusterRP/nysc_survey.survey_responses23.csv
Output directory: /Users/vimashafernando/Desktop/ClusterRP/outputs


## 1. Analysis specification

The lists below make the transformation reproducible. Variables are grouped using the types and clustering roles described in the supervisor plan. Fields not named in the core specification remain available in the cleaned export but do not silently enter the clustering matrix.

In [2]:
UNIVERSAL_VARIABLES = [
    "age_group", "gender", "district", "area_type", "moved_from_origin", "move_reason",
    "marital_status", "age_first_married", "spouse_activity", "has_children", "num_children",
    "father_edu", "mother_edu", "father_occ", "mother_occ", "household_size",
    "household_income", "household_workers", "household_unemployed", "primary_earner_status",
    "employment_network_quality", "edu_status", "reason_left_edu", "highest_edu_completed",
    "edu_end_year", "spell1_activity", "spell1_is_current", "spell1_contract",
    "spell2_activity", "spell2_is_current", "spell3_activity", "worked_while_studying",
    "edu_min_for_decent_job", "most_important_life_goal", "most_useful_quality_for_job",
    "ideal_work_type", "ideal_sector", "primary_income_source", "current_activity_status",
]

BRANCH_VARIABLES = {
    "D": [
        "plans_after_education", "already_job_searching", "re_entered_education",
        "reason_re_entered", "expected_highest_edu_d", "study_field_preference",
        "institution_type", "edu_funding_source", "social_capital_in_edu_access",
        "study_motivation_type", "autonomy_in_study", "competence_in_study_field",
        "relatedness_at_institution", "job_search_method_d",
    ],
    "E": [
        "desire_for_work", "active_search_4weeks", "search_method_primary", "available_to_start",
        "job_search_duration", "job_type_sought", "ever_refused_job", "job_acceptance_conditions",
        "has_minimum_wage", "minimum_wage_band", "num_applications", "num_interviews",
        "willing_to_relocate", "main_obstacle_finding_work", "past_edu_usefulness",
        "training_type_would_help", "future_planning_orientation", "knows_people_in_target_sector",
        "network_helpfulness", "income_urgency", "job_search_self_efficacy",
        "job_search_autonomy_orientation", "social_support_job_search",
    ],
    "F": [
        "work_arrangement", "enterprise_location_type", "enterprise_registered", "paid_taxes",
        "occupational_class", "contract_type", "contract_satisfaction", "has_second_job",
        "usual_hours_per_week", "actual_hours_last_week", "income_hours_preference",
        "hours_below_normal", "monthly_income_band", "job_satisfaction", "hopes_to_change_job",
        "job_security_12months", "insecurity_bothers", "willing_to_relocate_f",
        "plans_further_edu_f", "union_member", "received_training", "registered_as_jobseeker",
        "jobs_applied_before_hire", "interviews_before_hire", "ever_refused_job_f",
        "acceptance_conditions_search", "had_min_wage_during_search", "search_duration_before_hire",
        "obstacle_finding_this_job", "past_edu_useful_for_job", "employment_status",
        "job_recruitment_method", "enterprise_size", "reason_self_employed", "business_helpers",
        "business_funding_source", "business_profitability", "main_business_obstacle",
        "qualification_job_match", "sector_formality", "professional_network_growth",
        "work_autonomy", "competence_growth_at_work", "relatedness_at_work", "work_motivation_type",
    ],
    "G": [
        "reason_not_in_labour_force", "ever_worked", "wants_to_work_future", "wants_further_edu",
        "expected_highest_edu_g", "time_use_decision_maker", "labour_market_connection",
        "future_orientation_g", "employment_self_efficacy", "support_for_engagement",
    ],
}

MODEL_VARIABLES = list(dict.fromkeys(UNIVERSAL_VARIABLES + sum(BRANCH_VARIABLES.values(), [])))

COUNT_CONTINUOUS = {
    "age_first_married", "num_children", "household_size", "household_workers",
    "household_unemployed", "edu_end_year", "usual_hours_per_week", "actual_hours_last_week",
    "num_applications", "num_interviews", "jobs_applied_before_hire", "interviews_before_hire",
    "business_helpers",
}

ORDINAL_VARIABLES = {
    "age_group", "area_type", "father_edu", "mother_edu", "household_income",
    "employment_network_quality", "highest_edu_completed", "spell1_contract",
    "edu_min_for_decent_job", "expected_highest_edu_d", "study_motivation_type",
    "autonomy_in_study", "competence_in_study_field", "relatedness_at_institution",
    "job_search_duration", "job_acceptance_conditions", "minimum_wage_band", "income_urgency",
    "future_planning_orientation", "job_search_self_efficacy", "job_search_autonomy_orientation",
    "social_support_job_search", "contract_type", "contract_satisfaction", "monthly_income_band",
    "job_satisfaction", "job_security_12months", "enterprise_size", "business_profitability",
    "qualification_job_match", "sector_formality", "professional_network_growth", "work_autonomy",
    "competence_growth_at_work", "relatedness_at_work", "work_motivation_type",
    "expected_highest_edu_g", "labour_market_connection", "future_orientation_g",
    "employment_self_efficacy", "support_for_engagement",
}

BINARY_VARIABLES = {
    "moved_from_origin", "has_children", "spell1_is_current", "spell2_is_current",
    "worked_while_studying", "already_job_searching", "re_entered_education", "desire_for_work",
    "active_search_4weeks", "available_to_start", "ever_refused_job", "has_minimum_wage",
    "has_second_job", "hours_below_normal", "insecurity_bothers",
    "union_member", "received_training",
    "registered_as_jobseeker", "ever_refused_job_f", "had_min_wage_during_search", "ever_worked",
    "wants_to_work_future",
}

BRANCH_FROM_ACTIVITY = {
    "education": "D", "studying": "D",
    "unemployed": "E", "finished_plan_look": "E",
    "working": "F", "temp_absent": "F", "finished_plan_work": "F",
    "home_duties": "G", "other_inactive": "G", "inactive": "G",
}

SPECIAL_ORDINAL_MAPS = {
    "age_group": {
        "15-17": 1, "15–17": 1, "18-21": 2, "18–21": 2, "22-25": 3, "22–25": 3,
        "26-29": 4, "26–29": 4, "25-30": 4, "26-30": 4, "30-35": 5, "30–35": 5,
        "above-35": 6, "above 35": 6,
    },
    "area_type": {"rural": 1, "small_town": 2, "town": 2, "metro": 3, "metro_suburb": 3,
                  "large_city": 4, "city": 4, "urban": 4},
}

YES_VALUES = {"yes", "y", "true", "1", "1.0"}
NO_VALUES = {"no", "n", "false", "0", "0.0"}
MISSING_TOKENS = {"", "na", "n/a", "nan", "none", "null", "undefined", "missing"}

print(f"Specified clustering variables: {len(MODEL_VARIABLES)}")

Specified clustering variables: 131


## 2. Load and audit the raw export

The CSV contains several export generations: early records use top-level fields or `analysis.items[...]`, while later records use `answers.<question>.*`. The reconstruction step handles all three layouts and ignores audit/off-path answer payloads.

In [3]:
raw = pd.read_csv(INPUT_CSV, low_memory=False)

raw_audit = pd.Series({
    "rows": raw.shape[0],
    "columns": raw.shape[1],
    "duplicate_column_names": int(raw.columns.duplicated().sum()),
    "completely_empty_rows": int(raw.isna().all(axis=1).sum()),
    "memory_mb": round(raw.memory_usage(deep=True).sum() / 1024**2, 2),
})
display(raw_audit.to_frame("value"))

non_null_per_row = raw.notna().sum(axis=1)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(non_null_per_row, bins=30, color="#376996", edgecolor="white")
ax.set(title="Raw non-null cells per row", xlabel="Non-null cells", ylabel="Rows")
plt.show()

,value
rows,481.00
columns,2547.00
duplicate_column_names,0.00
completely_empty_rows,0.00
memory_mb,38.89


/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/664269962.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Reconstruct one canonical row per submission

Only active-path responses are used. Human-readable labels are retained in a parallel table for interpretation, while internal values are retained for correct ordinal handling.

In [4]:
def clean_scalar(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, str):
        value = re.sub(r"\s+", " ", value).strip()
        if value.lower() in MISSING_TOKENS:
            return pd.NA
    return value


def clean_series(series):
    return series.map(clean_scalar)


def first_existing(frame, columns):
    result = pd.Series(pd.NA, index=frame.index, dtype="object")
    for column in columns:
        if column in frame.columns:
            result = result.combine_first(clean_series(frame[column]))
    return result


def assign_where_missing(target, variable, values, mask=None):
    if variable is pd.NA or not isinstance(variable, str) or not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", variable):
        return
    values = clean_series(values)
    if variable not in target:
        target[variable] = pd.Series(pd.NA, index=raw.index, dtype="object")
    eligible = values.notna() & target[variable].isna()
    if mask is not None:
        eligible &= mask
    target.loc[eligible, variable] = values.loc[eligible]


def reconstruct_survey(frame):
    values = pd.DataFrame(index=frame.index)
    labels = pd.DataFrame(index=frame.index)
    metadata_rows = []

    # New export layout: answers.<question_id>.<attribute>
    answer_variable_columns = [c for c in frame if re.fullmatch(r"answers\.[^.]+\.variable", c)]
    for variable_column in answer_variable_columns:
        prefix = variable_column.rsplit(".", 1)[0]
        value_column, label_column = prefix + ".value", prefix + ".label"
        active_column = prefix + ".is_active_path"
        active = pd.Series(True, index=frame.index)
        if active_column in frame:
            active_text = frame[active_column].astype(str).str.strip().str.lower()
            active = ~active_text.isin({"false", "0", "no"})

        variables = clean_series(frame[variable_column])
        for variable in variables.dropna().astype(str).unique():
            mask = variables.eq(variable) & active
            if value_column in frame:
                assign_where_missing(values, variable, frame[value_column], mask)
            if label_column in frame:
                assign_where_missing(labels, variable, frame[label_column], mask)

        sample = variables.dropna()
        if not sample.empty:
            variable = str(sample.mode().iloc[0])
            metadata_rows.append({
                "variable": variable,
                "question_id": prefix.split(".")[1],
                "section": first_existing(frame, [prefix + ".section"]).dropna().mode().iloc[0]
                    if first_existing(frame, [prefix + ".section"]).notna().any() else pd.NA,
                "framework": first_existing(frame, [prefix + ".framework"]).dropna().mode().iloc[0]
                    if first_existing(frame, [prefix + ".framework"]).notna().any() else pd.NA,
                "question_text": first_existing(frame, [prefix + ".question_text"]).dropna().mode().iloc[0]
                    if first_existing(frame, [prefix + ".question_text"]).notna().any() else pd.NA,
            })

    # Intermediate export layout: analysis.items[index].<attribute>
    item_variable_columns = [c for c in frame if re.fullmatch(r"analysis\.items\[\d+\]\.variable", c)]
    for variable_column in item_variable_columns:
        prefix = variable_column.rsplit(".", 1)[0]
        variables = clean_series(frame[variable_column])
        for variable in variables.dropna().astype(str).unique():
            mask = variables.eq(variable)
            if prefix + ".value" in frame:
                assign_where_missing(values, variable, frame[prefix + ".value"], mask)
            if prefix + ".label" in frame:
                assign_where_missing(labels, variable, frame[prefix + ".label"], mask)

    # Explicit analysis dictionaries and legacy top-level survey fields.
    for column in [c for c in frame if c.startswith("analysis.values.")]:
        variable = column.removeprefix("analysis.values.")
        assign_where_missing(values, variable, frame[column])
    for column in [c for c in frame if c.startswith("analysis.labels.")]:
        variable = column.removeprefix("analysis.labels.")
        assign_where_missing(labels, variable, frame[column])
    for variable in MODEL_VARIABLES:
        if variable in frame:
            assign_where_missing(values, variable, frame[variable])

    metadata = pd.DataFrame(metadata_rows).drop_duplicates("variable") if metadata_rows else pd.DataFrame()
    return values, labels, metadata


canonical_values, canonical_labels, question_metadata = reconstruct_survey(raw)
print("Canonical variables recovered:", canonical_values.shape[1])
display(canonical_values.head())

/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2260493158.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  target[variable] = pd.Series(pd.NA, index=raw.index, dtype="object")
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2260493158.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  target[variable] = pd.Series(pd.NA, index=raw.index, dtype="object")
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2260493158.py:28: PerformanceWarning: DataFrame is highly fragmented.  This 

Canonical variables recovered: 162


/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2260493158.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  target[variable] = pd.Series(pd.NA, index=raw.index, dtype="object")
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2260493158.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  target[variable] = pd.Series(pd.NA, index=raw.index, dtype="object")


,age_group,gender,district,area_type,moved_from_origin,move_reason,marital_status,age_first_married,spouse_activity,has_children,num_children,father_edu,mother_edu,father_occ,mother_occ,household_size,household_income,household_workers,household_unemployed,primary_earner_status,employment_network_quality,edu_status,highest_edu_completed,edu_end_year,spell1_activity,spell1_is_current,spell2_activity,spell2_is_current,spell2_contract,spell2_satisfaction,spell2_end_reason,spell3_activity,worked_while_studying,work_while_study_type,work_while_study_paid,work_while_study_hours,work_while_study_motivation,edu_min_for_decent_job,most_important_life_goal,most_useful_quality_for_job,ideal_work_type,ideal_sector,primary_income_source,current_activity_status,reason_not_in_labour_force,ever_worked,wants_to_work_future,wants_further_edu,time_use_decision_maker,labour_market_connection,future_orientation_g,employment_self_efficacy,support_for_engagement,spell1_contract,spell1_satisfaction,spell1_end_reason,has_job_to_return,reason_temp_absence,enterprise_location_type,enterprise_registered,...,looking_while_absent,expected_highest_edu_g,reason_left_edu,desire_for_work,active_search_4weeks,wanted_to_work,reason_not_seeking,available_to_start,job_search_duration,job_type_sought,ever_refused_job,reason_refused_job,job_acceptance_conditions,has_minimum_wage,employment_service_help,activity_while_searching,num_applications,num_interviews,willing_to_relocate,main_obstacle_finding_work,past_edu_usefulness,training_type_would_help,plans_further_edu,expected_highest_edu_e,knows_people_in_target_sector,network_helpfulness,income_urgency,job_search_self_efficacy,job_search_autonomy_orientation,social_support_job_search,future_planning_orientation,plans_after_education,already_job_searching,re_entered_education,expected_highest_edu_d,study_field_preference,institution_type,edu_funding_source,social_capital_in_edu_access,study_motivation_type,autonomy_in_study,competence_in_study_field,relatedness_at_institution,reason_below_normal_hours,reason_wants_to_change,training_type_received,training_provider,training_duration,who_paid_training,reason_refused_job_f,job_recruitment_method,enterprise_size,gender_equality_at_work,search_method_primary,minimum_wage_band,job_search_method_d,reason_re_entered,expected_highest_edu_f,smoke_test,work_arrangement
0,18-21,male,Batticaloa,small_town,yes,<NA>,married,21.0,4.0,yes,1.0,2.0,2.0,3.0,4.0,2.0,3.0,2.0,1.0,no,3.0,left_early,1.0,2012.0,unemployed,current,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,no,<NA>,<NA>,<NA>,<NA>,4.0,2.0,3.0,3.0,10.0,4.0,working,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.0,no,...,<NA>,<NA>,3.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,wage
1,18-21,male,Batticaloa,small_town,yes,<NA>,married,21.0,4.0,yes,1.0,2.0,2.0,3.0,4.0,2.0,3.0,2.0,1.0,no,3.0,left_early,1.0,2012.0,unemployed,current,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,no,<NA>,<NA>,<NA>,<NA>,4.0,2.0,3.0,3.0,10.0,4.0,working,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.0,no,...,<NA>,<NA>,3.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,wage
2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>

## 4. Final active-branch resolution, quality rules, and de-duplication

The final analysis branch is authoritative. Newer application exports name the four branch sections C/D/E/F, whereas the research report names the same semantic branches D/E/F/G. The notebook translates them as `C→D`, `D→E`, `E→F`, and `F→G` before assigning the final branch.

`entry_branch` preserves the original C20 route for auditing. It is used only as a fallback when final-route metadata is absent. Consequently, a respondent who entered unemployment but was later routed to inactivity is analysed under G, while their entry branch remains E.

Quality filtering is conservative and auditable:

- known development/test/smoke submissions are excluded;
- an explicit non-complete status is excluded;
- records need a recognised final D/E/F/G route and at least 10 recovered survey answers;
- repeated session IDs are treated as re-submissions, keeping the latest record;
- legacy rows without a session ID remain distinct via their database `_id`.

The raw CSV is never modified.

In [5]:
respondent_id = first_existing(raw, ["respondent.session_id", "audit.original_meta.session_id", "_id"])
submitted_at = pd.to_datetime(
    first_existing(raw, ["submission.submitted_at", "audit.original_meta.timestamp", "_meta.timestamp", "createdAt", "timestamp"]),
    errors="coerce", utc=True,
)
status = first_existing(raw, ["submission.status", "audit.original_meta.completion_status"])
path_branch = first_existing(raw, ["path.branch", "audit.original_meta.branch", "_meta.branch"])

activity = clean_series(canonical_values.get("current_activity_status", pd.Series(pd.NA, index=raw.index)))

# Entry branch records where C20 initially routed the respondent. It is retained
# for auditing only and never overrides a valid final active branch.
entry_branch = activity.astype("string").str.lower().map(BRANCH_FROM_ACTIVITY)
entry_branch = entry_branch.combine_first(path_branch.astype("string").str.lower().map(BRANCH_FROM_ACTIVITY))

# Newer application exports use implementation sections C/D/E/F, while the
# research report names the same semantic branches D/E/F/G. During a
# mid-survey switch, path.active_branch can retain the gateway branch. The
# last branch-specific item in path.active_sections is therefore authoritative.
application_to_research = {"C": "D", "D": "E", "E": "F", "F": "G"}
application_active_raw = first_existing(raw, ["path.active_branch", "audit.original_meta.active_branch"])
application_active_branch = application_active_raw.astype("string").str.upper().map(application_to_research)
def ordered_active_section_columns(prefix):
    columns = [
        column for column in raw.columns
        if re.fullmatch(fr"{re.escape(prefix)}\[\d+\]", column)
    ]
    return sorted(columns, key=lambda column: int(re.search(r"\[(\d+)\]", column).group(1)))

def last_implementation_section(row):
    observed = [
        str(value).upper() for value in row
        if pd.notna(value) and str(value).upper() in application_to_research
    ]
    return observed[-1] if observed else pd.NA

path_active_section_columns = ordered_active_section_columns("path.active_sections")
audit_active_section_columns = ordered_active_section_columns("audit.original_meta.active_sections")
path_last_section = (
    raw[path_active_section_columns].apply(last_implementation_section, axis=1)
    if path_active_section_columns else pd.Series(pd.NA, index=raw.index, dtype="object")
)
audit_last_section = (
    raw[audit_active_section_columns].apply(last_implementation_section, axis=1)
    if audit_active_section_columns else pd.Series(pd.NA, index=raw.index, dtype="object")
)
application_last_section_raw = path_last_section.combine_first(audit_last_section)
application_last_section_branch = application_last_section_raw.map(application_to_research)
application_final_branch = application_last_section_branch.combine_first(application_active_branch)
application_path_disagreement = (
    application_last_section_branch.notna()
    & application_active_branch.notna()
    & application_last_section_branch.ne(application_active_branch)
)

# Legacy exports already stored research branch letters D/E/F/G.
legacy_active_raw = first_existing(raw, ["_meta.active_branch"])
legacy_final_branch = legacy_active_raw.astype("string").str.upper()
legacy_final_branch = legacy_final_branch.where(legacy_final_branch.isin(list("DEFG")))

# The final active analysis branch is authoritative. The entry branch is only a
# fallback for legacy rows that do not contain final-path metadata.
final_active_branch = application_final_branch.combine_first(legacy_final_branch)
branch = final_active_branch.combine_first(entry_branch)
branch_routing_source = pd.Series("entry_branch_fallback", index=raw.index, dtype="object")
branch_routing_source = branch_routing_source.mask(legacy_final_branch.notna(), "legacy_final_active_branch")
branch_routing_source = branch_routing_source.mask(application_final_branch.notna(), "translated_final_active_branch")
branch_changed = final_active_branch.notna() & entry_branch.notna() & final_active_branch.ne(entry_branch)
branch_path_disagreement = application_path_disagreement.fillna(False)
branch_integrity_issue = (
    application_final_branch.eq("F")
    & canonical_values.get("employment_status", pd.Series(pd.NA, index=raw.index)).isna()
)

test_columns = [c for c in ["test_data", "test_verify", "_test", "validation_test", "smoke_test"] if c in raw]
explicit_test = raw[test_columns].notna().any(axis=1) if test_columns else pd.Series(False, index=raw.index)
test_session = respondent_id.astype("string").str.contains(r"debug|test|smoke", case=False, na=False)
explicit_test = explicit_test | test_session
test_route = path_branch.astype("string").str.lower().isin({"debug", "debug_proxy", "smoke", "test"})
complete = status.isna() | status.astype("string").str.lower().eq("complete")

feature_count = canonical_values.notna().sum(axis=1)
age_token = clean_series(canonical_values.get("age_group", pd.Series(pd.NA, index=raw.index)))
age_token = age_token.astype("string").str.lower().str.replace("–", "-", regex=False).str.replace(" years", "", regex=False).str.strip()
allowed_age_groups = {"15-17", "18-21", "22-25", "25-30", "26-29", "26-30", "30-35", "1", "1.0", "2", "2.0", "3", "3.0", "4", "4.0", "5", "5.0"}
age_in_scope = age_token.isin(allowed_age_groups)

quality = pd.DataFrame({
    "respondent_id": respondent_id,
    "submitted_at": submitted_at,
    "entry_branch": entry_branch,
    "final_active_branch": final_active_branch,
    "branch": branch,
    "branch_routing_source": branch_routing_source,
    "branch_changed": branch_changed,
    "branch_path_disagreement": branch_path_disagreement,
    "branch_integrity_issue": branch_integrity_issue,
    "feature_count": feature_count,
    "explicit_test": explicit_test | test_route,
    "complete": complete,
    "age_in_scope": age_in_scope,
}, index=raw.index)
quality["keep_before_dedup"] = (
    quality["respondent_id"].notna()
    & ~quality["explicit_test"]
    & quality["complete"]
    & quality["age_in_scope"]
    & quality["branch"].isin(list("DEFG"))
    & quality["feature_count"].ge(MIN_ANSWERED_FEATURES)
)

eligible = quality.loc[quality["keep_before_dedup"]].sort_values(["respondent_id", "submitted_at"], na_position="first")
keep_indices = eligible.drop_duplicates("respondent_id", keep="last").index
quality["keep_final"] = quality.index.isin(keep_indices)

quality_summary = pd.Series({
    "raw_rows": len(raw),
    "known_test_rows": int(quality["explicit_test"].sum()),
    "explicitly_incomplete_rows": int((~quality["complete"]).sum()),
    "outside_target_age_15_35": int((~quality["age_in_scope"]).sum()),
    "unrecognised_or_missing_branch": int((~quality["branch"].isin(list("DEFG"))).sum()),
    "too_few_answers": int(quality["feature_count"].lt(MIN_ANSWERED_FEATURES).sum()),
    "eligible_before_dedup": int(quality["keep_before_dedup"].sum()),
    "duplicate_resubmissions_removed": int(quality["keep_before_dedup"].sum() - quality["keep_final"].sum()),
    "final_branch_overrides_retained": int((quality["branch_changed"] & quality["keep_final"]).sum()),
    "mid_survey_branch_switches_corrected": int((quality["branch_path_disagreement"] & quality["keep_final"]).sum()),
    "branch_integrity_issues_retained": int((quality["branch_integrity_issue"] & quality["keep_final"]).sum()),
    "entry_branch_fallback_retained": int((quality["branch_routing_source"].eq("entry_branch_fallback") & quality["keep_final"]).sum()),
    "final_rows": int(quality["keep_final"].sum()),
})
branch_correction_audit = pd.DataFrame({
    "respondent_id": respondent_id,
    "submitted_at": submitted_at,
    "entry_branch": entry_branch,
    "stale_active_branch": application_active_branch,
    "last_active_section": application_last_section_raw,
    "corrected_final_branch": application_final_branch,
    "activity_status": activity,
}, index=raw.index).loc[quality["keep_final"] & quality["branch_path_disagreement"]]
branch_correction_audit.to_csv(OUTPUT_DIR / "nysc_survey_branch_path_corrections.csv", index=False)
branch_integrity_audit = pd.DataFrame({
    "respondent_id": respondent_id,
    "submitted_at": submitted_at,
    "final_branch": branch,
    "activity_status": activity,
    "employment_status": canonical_values.get("employment_status", pd.Series(pd.NA, index=raw.index)),
}, index=raw.index).loc[quality["keep_final"] & quality["branch_integrity_issue"]]
branch_integrity_audit.to_csv(OUTPUT_DIR / "nysc_survey_branch_integrity_issues.csv", index=False)
display(quality_summary.to_frame("count"))
display(quality.loc[quality["keep_final"], "branch"].value_counts().sort_index().to_frame("rows"))
display(branch_correction_audit)
display(branch_integrity_audit)

,count
raw_rows,481
known_test_rows,11
explicitly_incomplete_rows,0
outside_target_age_15_35,30
unrecognised_or_missing_branch,10
too_few_answers,14
eligible_before_dedup,443
duplicate_resubmissions_removed,51
final_branch_overrides_retained,31
mid_survey_branch_switches_corrected,11


,rows
branch,
D,195
E,38
F,90
G,69


,respondent_id,submitted_at,entry_branch,stale_active_branch,last_active_section,corrected_final_branch,activity_status
48,nysc-31cab333-afc9-48c5-baec-8ba48e14defb,2026-05-09 09:18:15.305000+00:00,E,E,F,G,unemployed
54,nysc-9a123603-4572-4552-aceb-dfb609277901,2026-05-09 13:55:15.207000+00:00,F,F,F,G,temp_absent
57,nysc-f3fcb021-3af8-4a20-bc37-5a8c05409f8d,2026-05-09 14:48:56.381000+00:00,F,F,F,G,temp_absent
75,nysc-7461ca22-c4f5-449e-a85b-1228930b73ec,2026-08-04 12:47:31.893000+00:00,E,E,F,G,unemployed
89,nysc-5242257e-4708-442a-a41f-1081109194a9,2026-08-05 12:21:18.923000+00:00,F,F,F,G,finished_plan_work
120,nysc-727a27bc-7f06-4df3-9d33-9e89832b63fd,2026-08-08 03:21:23.379000+00:00,E,E,F,G,unemployed
121,nysc-9ca93a7c-2fdc-4cc9-9141-a79a6e78d214,2026-08-08 05:28:17.032000+00:00,E,E,F,G,finished_plan_look
122,nysc-b8fb3c50-5810-4e71-9133-6fa7cf5cb36f,2026-08-08 10:56:03.212000+00:00,F,F,F,G,finished_plan_work
135,nysc-9cb7eb74-e559-499a-bb4f-b009c74c71d9,2026-08-09 10:24:00.479000+00:00,E,E,F,G,unemployed
142,nysc-131a2b60-1a70-4974-9dc0-8a1eaf51befd,2026-08-09 16:04:35.053000+00:00,F,F,D,E,temp_absent


,respondent_id,submitted_at,final_branch,activity_status,employment_status
409,nysc-d7da89a3-4c8d-4e5a-83ff-b2793774fdce,2026-08-13 16:58:08.459000+00:00,F,working,<NA>


## 5. Create cleaned value and label datasets

`cleaned_values` preserves internal survey codes for transformation. `cleaned_labels` preserves readable response labels for EDA and cluster profiling. Off-path variables remain null here; they are marked explicitly only at the modeling stage.

In [6]:
kept = quality["keep_final"]
admin = quality.loc[kept, ["respondent_id", "submitted_at", "entry_branch", "branch", "branch_routing_source", "branch_changed", "branch_path_disagreement", "branch_integrity_issue", "feature_count"]].reset_index(drop=True)
value_body = canonical_values.loc[kept].reset_index(drop=True)
label_body = canonical_labels.loc[kept].reindex(columns=value_body.columns).reset_index(drop=True)

cleaned_values = pd.concat([admin, value_body], axis=1)
cleaned_labels = pd.concat([admin, label_body], axis=1)

cleaned_values.to_csv(OUTPUT_DIR / "nysc_survey_cleaned_values.csv", index=False)
cleaned_labels.to_csv(OUTPUT_DIR / "nysc_survey_cleaned_labels.csv", index=False)
quality.to_csv(OUTPUT_DIR / "nysc_survey_quality_audit.csv", index_label="raw_row")

print("Cleaned shape:", cleaned_values.shape)
display(cleaned_values.head())

Cleaned shape: (392, 171)


,respondent_id,submitted_at,entry_branch,branch,branch_routing_source,branch_changed,branch_path_disagreement,branch_integrity_issue,feature_count,age_group,gender,district,area_type,moved_from_origin,move_reason,marital_status,age_first_married,spouse_activity,has_children,num_children,father_edu,mother_edu,father_occ,mother_occ,household_size,household_income,household_workers,household_unemployed,primary_earner_status,employment_network_quality,edu_status,highest_edu_completed,edu_end_year,spell1_activity,spell1_is_current,spell2_activity,spell2_is_current,spell2_contract,spell2_satisfaction,spell2_end_reason,spell3_activity,worked_while_studying,work_while_study_type,work_while_study_paid,work_while_study_hours,work_while_study_motivation,edu_min_for_decent_job,most_important_life_goal,most_useful_quality_for_job,ideal_work_type,ideal_sector,primary_income_source,current_activity_status,reason_not_in_labour_force,ever_worked,wants_to_work_future,wants_further_edu,time_use_decision_maker,labour_market_connection,future_orientation_g,...,looking_while_absent,expected_highest_edu_g,reason_left_edu,desire_for_work,active_search_4weeks,wanted_to_work,reason_not_seeking,available_to_start,job_search_duration,job_type_sought,ever_refused_job,reason_refused_job,job_acceptance_conditions,has_minimum_wage,employment_service_help,activity_while_searching,num_applications,num_interviews,willing_to_relocate,main_obstacle_finding_work,past_edu_usefulness,training_type_would_help,plans_further_edu,expected_highest_edu_e,knows_people_in_target_sector,network_helpfulness,income_urgency,job_search_self_efficacy,job_search_autonomy_orientation,social_support_job_search,future_planning_orientation,plans_after_education,already_job_searching,re_entered_education,expected_highest_edu_d,study_field_preference,institution_type,edu_funding_source,social_capital_in_edu_access,study_motivation_type,autonomy_in_study,competence_in_study_field,relatedness_at_institution,reason_below_normal_hours,reason_wants_to_change,training_type_received,training_provider,training_duration,who_paid_training,reason_refused_job_f,job_recruitment_method,enterprise_size,gender_equality_at_work,search_method_primary,minimum_wage_band,job_search_method_d,reason_re_entered,expected_highest_edu_f,smoke_test,work_arrangement
0,69fac94d4a154e3795d7a181,2026-05-06 04:53:33.973000+00:00,F,F,entry_branch_fallback,False,False,False,77,18-21,male,Batticaloa,small_town,yes,<NA>,married,21.0,4.0,yes,1.0,2.0,2.0,3.0,4.0,2.0,3.0,2.0,1.0,no,3.0,left_early,1.0,2012.0,unemployed,current,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,no,<NA>,<NA>,<NA>,<NA>,4.0,2.0,3.0,3.0,10.0,4.0,working,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,3.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,wage
1,69fac9d34a154e3795d7a182,2026-05-06 04:55:47.361000+00:00,F,F,entry_branch_fallback,False,False,False,77,18-21,male,Batticaloa,small_town,yes,<NA>,married,21.0,4.0,yes,1.0,2.0,2.0,3.0,4.0,2.0,3.0,2.0,1.0,no,3.0,left_early,1.0,2012.0,unemployed,current,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,no,<NA>,<NA>,<NA>,<NA>,4.0,2.0,3.0,3.0,10.0,4.0,working,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,3.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,wage
2,69fb3d4fab1bd766333f2f36,2026-05-06 13:08:31.211000+00:00,F,F,entry_branch_fallback,False,False,False,76,22-25,female,Hambantota,small_town,no,3.0,married,5.0,2.0,no,<NA>,2.0,3.0,4.0,4.0,23.0,2.0,25.0,0.0,yes,1.0,completed,4.0,6.0,wage_work,current,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,no,<NA>,<NA>,<NA>,<NA>,4.0,3.0,7.0,4.0,4.0,4

## 6. Missingness and structural applicability

Off-branch fields are structural nulls—not respondent omissions. In the analysis-ready export they are recorded as `not_applicable`; missing values in fields that should have been available remain `missing`. Numeric modeling features also receive `__applicable` indicators so imputation cannot erase this distinction.

In [7]:
PRESENT_MODEL_VARIABLES = [v for v in MODEL_VARIABLES if v in value_body and value_body[v].notna().any()]
VARIABLE_BRANCH = {v: b for b, variables in BRANCH_VARIABLES.items() for v in variables}

analysis_ready = admin.copy()
for variable in PRESENT_MODEL_VARIABLES:
    series = clean_series(label_body[variable]).combine_first(clean_series(value_body[variable]))
    applicable_branch = VARIABLE_BRANCH.get(variable)
    if applicable_branch:
        # Preserve an observed gateway answer when it was genuinely on the
        # active path before a mid-survey branch switch. Only an absent value
        # from another final branch is structurally not applicable.
        off_path = admin["branch"].ne(applicable_branch) & series.isna()
        series = series.mask(off_path, "not_applicable")
        series = series.fillna("missing")
    analysis_ready[variable] = series

analysis_ready.to_csv(OUTPUT_DIR / "nysc_survey_analysis_ready.csv", index=False)

missing_rows = []
for variable in PRESENT_MODEL_VARIABLES:
    applicable_branch = VARIABLE_BRANCH.get(variable)
    applicable = pd.Series(True, index=admin.index) if not applicable_branch else admin["branch"].eq(applicable_branch)
    missing_rows.append({
        "variable": variable,
        "applicable_branch": applicable_branch or "all",
        "applicable_rows": int(applicable.sum()),
        "true_missing": int((applicable & value_body[variable].isna()).sum()),
        "structural_not_applicable": int((~applicable).sum()),
        "observed": int((applicable & value_body[variable].notna()).sum()),
    })
missing_report = pd.DataFrame(missing_rows)
missing_report["true_missing_pct"] = np.where(
    missing_report["applicable_rows"] > 0,
    100 * missing_report["true_missing"] / missing_report["applicable_rows"],
    np.nan,
)
missing_report.to_csv(OUTPUT_DIR / "nysc_survey_missingness.csv", index=False)
display(missing_report.sort_values("true_missing_pct", ascending=False).head(20))

plot_missing = missing_report.nlargest(20, "true_missing_pct").sort_values("true_missing_pct")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(plot_missing["variable"], plot_missing["true_missing_pct"], color="#DB6B52")
ax.set(title="Highest genuine missingness among applicable rows", xlabel="Missing (%)")
plt.tight_layout()
plt.show()

,variable,applicable_branch,applicable_rows,true_missing,structural_not_applicable,observed,true_missing_pct
30,spell3_activity,all,392,386,0,6,98.469388
27,spell1_contract,all,392,380,0,12,96.938776
22,reason_left_edu,all,392,376,0,16,95.918367
29,spell2_is_current,all,392,376,0,16,95.918367
28,spell2_activity,all,392,376,0,16,95.918367
10,num_children,all,392,371,0,21,94.642857
76,work_arrangement,F,90,85,302,5,94.444444
7,age_first_married,all,392,348,0,44,88.775510
8,spouse_activity,all,392,348,0,44,88.775510
26,spell1_is_current,all,392,334,0,58,85.204082


/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/3574383688.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Type-aware transformation

- Count/continuous variables and documented ordinal codes are converted to numeric values.
- Binary fields are mapped to 0/1.
- Nominal variables are one-hot encoded.
- Numeric missing values use median imputation; categorical missing values use a literal `missing` level.
- Standardization is fitted after imputation.

No row is deleted as an automatic outlier treatment. Survey codes and legitimate extremes are meaningful for clustering, so the IQR table below is a diagnostic for review rather than a deletion rule.

In [8]:
def to_ordinal(series, variable):
    cleaned = clean_series(series)
    numeric = pd.to_numeric(cleaned, errors="coerce")
    if variable in SPECIAL_ORDINAL_MAPS:
        text = cleaned.astype("string").str.lower().str.replace(" years", "", regex=False).str.strip()
        mapped = text.map({str(k).lower(): v for k, v in SPECIAL_ORDINAL_MAPS[variable].items()})
        numeric = numeric.combine_first(mapped)
    return numeric.astype(float)


def to_binary(series):
    cleaned = clean_series(series)
    numeric = pd.to_numeric(cleaned, errors="coerce").where(lambda x: x.isin([0, 1]))
    text = cleaned.astype("string").str.lower().str.strip()
    mapped = text.map({
        **{x: 1 for x in YES_VALUES}, **{x: 0 for x in NO_VALUES},
        "current": 1, "ended": 0,
    })
    return numeric.combine_first(mapped).astype(float)


def prepare_model_frame(values_frame, admin_frame, variables):
    model = pd.DataFrame(index=values_frame.index)
    numeric_columns, nominal_columns, conversion_report = [], [], []

    for variable in variables:
        raw_series = values_frame[variable]
        branch_required = VARIABLE_BRANCH.get(variable)
        applicable = (
            pd.Series(True, index=values_frame.index)
            if not branch_required
            else admin_frame["branch"].eq(branch_required) | raw_series.notna()
        )

        if variable in BINARY_VARIABLES:
            converted = to_binary(raw_series).where(applicable)
            model[variable] = converted
            numeric_columns.append(variable)
        elif variable in COUNT_CONTINUOUS or variable in ORDINAL_VARIABLES:
            converted = to_ordinal(raw_series, variable).where(applicable)
            model[variable] = converted
            numeric_columns.append(variable)
        else:
            converted = clean_series(raw_series).astype("string")
            converted = converted.where(applicable, "not_applicable").fillna("missing").astype(str)
            model[variable] = converted
            nominal_columns.append(variable)

        observed = raw_series.notna() & applicable
        if variable in numeric_columns:
            failed = observed & model[variable].isna()
            conversion_report.append({
                "variable": variable,
                "observed_before_conversion": int(observed.sum()),
                "unmapped_numeric_values": int(failed.sum()),
            })

        if branch_required:
            indicator = variable + "__applicable"
            model[indicator] = applicable.astype(float)
            numeric_columns.append(indicator)

    # Remove fields with no usable observed value. Applicability indicators remain.
    all_missing_numeric = [c for c in numeric_columns if model[c].notna().sum() == 0]
    model = model.drop(columns=all_missing_numeric)
    numeric_columns = [c for c in numeric_columns if c not in all_missing_numeric]
    nominal_columns = [c for c in nominal_columns if model[c].nunique(dropna=True) > 1]
    model = model[numeric_columns + nominal_columns]

    return model, numeric_columns, nominal_columns, pd.DataFrame(conversion_report), all_missing_numeric


model_frame, numeric_columns, nominal_columns, conversion_report, dropped_all_missing = prepare_model_frame(
    value_body, admin, PRESENT_MODEL_VARIABLES
)

conversion_report.to_csv(OUTPUT_DIR / "nysc_survey_numeric_conversion_audit.csv", index=False)
print("Numeric features before expansion:", len(numeric_columns))
print("Nominal features before one-hot encoding:", len(nominal_columns))
print("All-missing specified fields excluded:", dropped_all_missing)
display(conversion_report.query("unmapped_numeric_values > 0").head(20))

/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2408534580.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model[indicator] = applicable.astype(float)
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2408534580.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model[variable] = converted
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2408534580.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which 

Numeric features before expansion: 168
Nominal features before one-hot encoding: 55
All-missing specified fields excluded: []


,variable,observed_before_conversion,unmapped_numeric_values


In [9]:
outlier_rows = []
for variable in [c for c in numeric_columns if not c.endswith("__applicable")]:
    values = model_frame[variable].dropna()
    if values.nunique() < 5:
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({
        "variable": variable, "q1": q1, "q3": q3, "lower_fence": lower,
        "upper_fence": upper, "flagged_rows": int(((values < lower) | (values > upper)).sum()),
    })
outlier_report = pd.DataFrame(outlier_rows)
outlier_report.to_csv(OUTPUT_DIR / "nysc_survey_outlier_diagnostics.csv", index=False)
display(outlier_report.sort_values("flagged_rows", ascending=False).head(15) if not outlier_report.empty else outlier_report)

,variable,q1,q3,lower_fence,upper_fence,flagged_rows
0,age_group,3.0,3.0,3.0,3.0,139
10,edu_min_for_decent_job,4.0,5.0,2.5,6.5,62
5,household_income,3.0,5.0,0.0,8.0,52
4,household_size,4.0,5.0,2.5,6.5,43
2,father_edu,3.0,6.0,-1.5,10.5,19
18,contract_type,5.0,6.0,3.5,7.5,18
3,mother_edu,3.0,6.0,-1.5,10.5,16
27,work_motivation_type,1.0,2.0,-0.5,3.5,14
23,jobs_applied_before_hire,2.0,10.0,-10.0,22.0,14
25,enterprise_size,4.0,5.0,2.5,6.5,9


## 8. Build the standardized full-sample feature matrix and PCA representation

In [10]:
def fit_transform_features(model, numeric, nominal, ids, prefix):
    transformers = []
    if numeric:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ])
        transformers.append(("num", numeric_pipeline, numeric))
    if nominal:
        nominal_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])
        transformers.append(("cat", nominal_pipeline, nominal))

    transformer = ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)
    matrix = transformer.fit_transform(model)
    feature_names = transformer.get_feature_names_out()
    feature_frame = pd.DataFrame(matrix, columns=feature_names, index=model.index)

    pca = PCA(n_components=PCA_VARIANCE_TARGET, svd_solver="full", random_state=RANDOM_STATE)
    pca_matrix = pca.fit_transform(feature_frame)
    pca_columns = [f"PC{i + 1}" for i in range(pca_matrix.shape[1])]
    pca_frame = pd.DataFrame(pca_matrix, columns=pca_columns, index=model.index)

    id_frame = ids.reset_index(drop=True)
    feature_out = pd.concat([id_frame, feature_frame.reset_index(drop=True)], axis=1)
    pca_out = pd.concat([id_frame, pca_frame.reset_index(drop=True)], axis=1)
    feature_out.to_csv(OUTPUT_DIR / f"{prefix}_ml_ready.csv", index=False)
    pca_out.to_csv(OUTPUT_DIR / f"{prefix}_pca85.csv", index=False)

    pca_summary = pd.DataFrame({
        "component": pca_columns,
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_variance": np.cumsum(pca.explained_variance_ratio_),
    })
    pca_summary.to_csv(OUTPUT_DIR / f"{prefix}_pca_summary.csv", index=False)
    return transformer, feature_frame, pca, pca_frame, pca_summary


full_transformer, full_features, full_pca, full_pca_frame, full_pca_summary = fit_transform_features(
    model_frame, numeric_columns, nominal_columns, admin[["respondent_id", "branch"]], "nysc_survey_full"
)

print("Standardized/one-hot feature matrix:", full_features.shape)
print("PCA representation:", full_pca_frame.shape)
print("Variance retained:", f"{full_pca.explained_variance_ratio_.sum():.2%}")
display(full_pca_summary.tail())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(full_pca_summary["component"], full_pca_summary["cumulative_variance"], marker="o", ms=3)
axes[0].axhline(PCA_VARIANCE_TARGET, color="crimson", linestyle="--", label="85% target")
axes[0].set(title="Cumulative PCA variance", xlabel="Component", ylabel="Cumulative variance")
axes[0].tick_params(axis="x", labelrotation=90)
axes[0].legend()

if full_pca_frame.shape[1] >= 2:
    colors = admin["branch"].map({"D": "#377eb8", "E": "#e41a1c", "F": "#4daf4a", "G": "#984ea3"})
    axes[1].scatter(full_pca_frame["PC1"], full_pca_frame["PC2"], c=colors, alpha=0.65, s=20)
    axes[1].set(title="PCA preview by survey branch", xlabel="PC1", ylabel="PC2")
plt.tight_layout()
plt.show()

Standardized/one-hot feature matrix: (392, 621)
PCA representation: (392, 36)
Variance retained: 85.32%


,component,explained_variance_ratio,cumulative_variance
31,PC32,0.004846,0.835219
32,PC33,0.004710,0.839929
33,PC34,0.004568,0.844497
34,PC35,0.004410,0.848907
35,PC36,0.004256,0.853162


/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_2980/2800268865.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Build branch-specific matrices

Each branch transformer is fitted independently and only sees the universal variables plus that branch's variables. This avoids feeding structurally irrelevant fields into within-branch clustering.

In [11]:
branch_summary_rows = []
for branch_code in "DEFG":
    mask = admin["branch"].eq(branch_code)
    branch_values = value_body.loc[mask].reset_index(drop=True)
    branch_admin = admin.loc[mask].reset_index(drop=True)
    branch_vars = [v for v in UNIVERSAL_VARIABLES + BRANCH_VARIABLES[branch_code]
                   if v in branch_values and branch_values[v].notna().any()]

    branch_model, branch_numeric, branch_nominal, _, branch_dropped = prepare_model_frame(
        branch_values, branch_admin, branch_vars
    )
    # Applicability is constant inside a branch and carries no information.
    applicable_columns = [c for c in branch_model if c.endswith("__applicable")]
    branch_model = branch_model.drop(columns=applicable_columns)
    branch_numeric = [c for c in branch_numeric if c not in applicable_columns]

    branch_analysis = analysis_ready.loc[mask, ["respondent_id", "entry_branch", "branch", "branch_changed"] + branch_vars].reset_index(drop=True)
    branch_analysis.to_csv(OUTPUT_DIR / f"nysc_survey_branch_{branch_code}_analysis_ready.csv", index=False)

    if len(branch_model) >= 3 and (branch_numeric or branch_nominal):
        _, b_features, b_pca, b_pca_frame, _ = fit_transform_features(
            branch_model, branch_numeric, branch_nominal,
            branch_admin[["respondent_id", "branch"]], f"nysc_survey_branch_{branch_code}"
        )
        branch_summary_rows.append({
            "branch": branch_code, "rows": len(branch_model), "input_variables": len(branch_vars),
            "encoded_features": b_features.shape[1], "pca_components": b_pca_frame.shape[1],
            "pca_variance_retained": b_pca.explained_variance_ratio_.sum(),
        })

branch_summary = pd.DataFrame(branch_summary_rows)
branch_summary.to_csv(OUTPUT_DIR / "nysc_survey_branch_preprocessing_summary.csv", index=False)
display(branch_summary)

,branch,rows,input_variables,encoded_features,pca_components,pca_variance_retained
0,D,195,48,204,33,0.853072
1,E,38,60,205,20,0.864054
2,F,90,84,284,34,0.852167
3,G,69,49,200,24,0.851443


## 10. Feature manifest and reproducibility report

In [12]:
manifest_rows = []
for variable in PRESENT_MODEL_VARIABLES:
    if variable in COUNT_CONTINUOUS:
        variable_type = "count_or_continuous"
        encoding = "median imputation + StandardScaler"
    elif variable in ORDINAL_VARIABLES:
        variable_type = "ordinal"
        encoding = "ordered survey code + median imputation + StandardScaler"
    elif variable in BINARY_VARIABLES:
        variable_type = "binary"
        encoding = "0/1 + median imputation + StandardScaler"
    else:
        variable_type = "nominal"
        encoding = "explicit missing/not_applicable + one-hot encoding"
    meta = question_metadata.loc[question_metadata.get("variable", pd.Series(dtype=object)).eq(variable)]
    manifest_rows.append({
        "variable": variable,
        "type": variable_type,
        "applicable_branch": VARIABLE_BRANCH.get(variable, "all"),
        "encoding": encoding,
        "question_id": meta["question_id"].iloc[0] if not meta.empty else pd.NA,
        "section": meta["section"].iloc[0] if not meta.empty else pd.NA,
        "framework": meta["framework"].iloc[0] if not meta.empty else pd.NA,
        "question_text": meta["question_text"].iloc[0] if not meta.empty else pd.NA,
    })
feature_manifest = pd.DataFrame(manifest_rows)
feature_manifest.to_csv(OUTPUT_DIR / "nysc_survey_feature_manifest.csv", index=False)

run_summary = {
    "input_file": str(INPUT_CSV.resolve()),
    "raw_shape": list(raw.shape),
    "cleaned_shape": list(cleaned_values.shape),
    "quality": {k: int(v) for k, v in quality_summary.items()},
    "branch_counts": {str(k): int(v) for k, v in admin["branch"].value_counts().sort_index().items()},
    "model_variables_present": len(PRESENT_MODEL_VARIABLES),
    "encoded_feature_count": int(full_features.shape[1]),
    "pca_components": int(full_pca_frame.shape[1]),
    "pca_variance_retained": float(full_pca.explained_variance_ratio_.sum()),
    "random_state": RANDOM_STATE,
    "pca_variance_target": PCA_VARIANCE_TARGET,
}
with open(OUTPUT_DIR / "nysc_survey_preprocessing_summary.json", "w", encoding="utf-8") as handle:
    json.dump(run_summary, handle, indent=2)

display(feature_manifest.head(20))
display(pd.Series(run_summary).to_frame("value"))

,variable,type,applicable_branch,encoding,question_id,section,framework,question_text
0,age_group,ordinal,all,ordered survey code + median imputation + Stan...,A1,A,SWTS,What is your age?
1,gender,nominal,all,explicit missing/not_applicable + one-hot enco...,A2,A,SWTS,What is your gender?
2,district,nominal,all,explicit missing/not_applicable + one-hot enco...,A3,A,SWTS,Which district do you currently live in?
3,area_type,ordinal,all,ordered survey code + median imputation + Stan...,A4,A,SWTS,How would you describe your current place of r...
4,moved_from_origin,binary,all,0/1 + median imputation + StandardScaler,A5,A,SWTS,Do you live in the same area where you grew up?
5,move_reason,nominal,all,explicit missing/not_applicable + one-hot enco...,A6,A,SWTS,What was the main reason you moved to your cur...
6,marital_status,nominal,all,explicit missing/not_applicable + one-hot enco...,A7,A,SWTS,What is your current marital status?
7,age_first_married,count_or_continuous,all,median imputation + StandardScaler,A8,A,SWTS,At what age were you first married?
8,spouse_activity,nominal,all,explicit missing/not_applicable + one-hot enco...,A9,A,SWTS,What does your spouse or partner currently do?
9,has_children,binary,all,0/1 + median imputation + StandardScaler,A10,A,SWTS,Do you have any children?


,value
input_file,/Users/vimashafernando/Desktop/ClusterRP/nysc_...
raw_shape,"[481, 2547]"
cleaned_shape,"[392, 171]"
quality,"{'raw_rows': 481, 'known_test_rows': 11, 'expl..."
branch_counts,"{'D': 195, 'E': 38, 'F': 90, 'G': 69}"
model_variables_present,131
encoded_feature_count,621
pca_components,36
pca_variance_retained,0.853162
random_state,42


## Output guide

- `nysc_survey_cleaned_values.csv`: canonical internal response values; best for reproducible transformations.
- `nysc_survey_cleaned_labels.csv`: readable response labels; best for EDA and cluster interpretation.
- `nysc_survey_analysis_ready.csv`: selected variables with explicit `missing` and `not_applicable` states.
- `nysc_survey_full_ml_ready.csv`: imputed, standardized, one-hot encoded full-sample matrix.
- `nysc_survey_full_pca85.csv`: full-sample PCA matrix retaining at least 85% variance.
- `nysc_survey_branch_[D–G]_*`: branch-specific analysis, ML, PCA, and PCA-summary files.
- audit files: quality decisions, missingness, numeric conversion, outlier flags, feature manifest, and run summary.

For later clustering, use the `*_ml_ready.csv` matrices with algorithms that operate well in the expanded standardized space, or the `*_pca85.csv` matrices when noise reduction is desired. Keep `respondent_id` and `branch` as identifiers only—never as clustering features.